# Notebook 08C H=8/H=16 Deep Learning

Manual Kaggle UI notebook for the controlled 8-configuration H=8/H=16 run.

This notebook is guarded. It will stop before training unless `APPROVED_EXECUTION = True` is changed manually after the required dataset, GPU, Internet, and ClearML secrets are confirmed in the Kaggle UI.


In [ ]:
from pathlib import Path
import json, os, time, math, zipfile, traceback, socket

APPROVED_EXECUTION = False
CLEARML_PROJECT = "PV_ramps_forecasting_MG_VF/08_DeepLearning"
CLEARML_TASK_NAME_PREFIX = "NOTEBOOK_08C_H8_H16_deep_learning"
OUTPUT_DIR = Path("/kaggle/working/outputs/08_deep_learning_models")
PLOTS_DIR = OUTPUT_DIR / "plots"
HISTORY_DIR = OUTPUT_DIR / "training_history"
RUN_STATE_DIR = OUTPUT_DIR / "run_state"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
for folder in [OUTPUT_DIR, PLOTS_DIR, HISTORY_DIR, RUN_STATE_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

OFFICIAL_CONFIGS = [
    {"resolution": "one_hour", "dataset_variant": "selected_physics", "model": "LSTM", "lookback": 16, "horizon": 8, "priority_order": 1},
    {"resolution": "one_minute", "dataset_variant": "selected_physics", "model": "LSTM", "lookback": 16, "horizon": 8, "priority_order": 2},
    {"resolution": "one_minute", "dataset_variant": "selected_physics", "model": "GRU", "lookback": 16, "horizon": 8, "priority_order": 3},
    {"resolution": "one_minute", "dataset_variant": "selected_physics", "model": "BiLSTM", "lookback": 16, "horizon": 8, "priority_order": 4},
    {"resolution": "one_hour", "dataset_variant": "selected_physics", "model": "LSTM", "lookback": 32, "horizon": 16, "priority_order": 5},
    {"resolution": "one_minute", "dataset_variant": "selected_physics", "model": "LSTM", "lookback": 32, "horizon": 16, "priority_order": 6},
    {"resolution": "one_minute", "dataset_variant": "selected_physics", "model": "GRU", "lookback": 32, "horizon": 16, "priority_order": 7},
    {"resolution": "one_minute", "dataset_variant": "selected_physics", "model": "BiLSTM", "lookback": 32, "horizon": 16, "priority_order": 8},
]
OPTIONAL_CONFIGS_DISABLED = [
    {"resolution": "one_hour", "dataset_variant": "selected_physics", "model": "GRU", "lookback": 16, "horizon": 8, "enabled": False},
    {"resolution": "one_hour", "dataset_variant": "selected_physics", "model": "GRU", "lookback": 32, "horizon": 16, "enabled": False},
]
assert len(OFFICIAL_CONFIGS) == 8
assert all(c["horizon"] in {8, 16} for c in OFFICIAL_CONFIGS)
assert all(c["lookback"] in {16, 32} for c in OFFICIAL_CONFIGS)
assert all(not c["enabled"] for c in OPTIONAL_CONFIGS_DISABLED)
print("08C official configs enabled:", len(OFFICIAL_CONFIGS))
print("Optional configs disabled:", len(OPTIONAL_CONFIGS_DISABLED))
print("APPROVED_EXECUTION:", APPROVED_EXECUTION)


In [ ]:
if not APPROVED_EXECUTION:
    raise SystemExit(
        "APPROVED_EXECUTION is False. This is intentional: no training can start until the user manually approves execution."
    )


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as exc:
    raise RuntimeError("TensorFlow/Keras is required for the Kaggle GPU run.") from exc

try:
    import joblib
except Exception as exc:
    raise RuntimeError("joblib is required to load train-only scalers.") from exc

REQUIRED_CLEARML_KEYS = [
    "CLEARML_API_ACCESS_KEY",
    "CLEARML_API_SECRET_KEY",
    "CLEARML_API_HOST",
    "CLEARML_WEB_HOST",
    "CLEARML_FILES_HOST",
]

def get_secret_from_kaggle(name):
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        value = client.get_secret(name)
        return value
    except Exception:
        return None

def clearml_secret_preflight():
    statuses = []
    for key in REQUIRED_CLEARML_KEYS:
        value = os.environ.get(key)
        source = "environment"
        if not value:
            secret_value = get_secret_from_kaggle(key)
            if secret_value:
                os.environ[key] = secret_value
                value = secret_value
                source = "kaggle_secret"
        statuses.append({"key": key, "status": "SET" if value else "MISSING", "source": source if value else "none"})
    missing = [s["key"] for s in statuses if s["status"] != "SET"]
    pd.DataFrame(statuses).to_csv(OUTPUT_DIR / "clearml_secret_status.csv", index=False)
    if missing:
        raise RuntimeError(f"Missing ClearML secrets before training: {missing}")
    return statuses

def internet_check(host="api.clear.ml", port=443, timeout=5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return {"internet_ok": True, "host": host, "port": port, "error": ""}
    except Exception as exc:
        return {"internet_ok": False, "host": host, "port": port, "error": str(exc)}

def gpu_check():
    gpus = tf.config.list_physical_devices("GPU")
    return {"gpu_available": bool(gpus), "gpu_count": len(gpus), "gpu_names": [str(g) for g in gpus]}

def resolve_input_root():
    candidates = [
        Path("/kaggle/input/pv-ramp-notebook-08c-h8-h16-inputs/outputs"),
        Path("/kaggle/input/pv-ramp-notebook-08b-h1-inputs/outputs"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find expected Kaggle input outputs/ root for Notebook 08C.")

secret_status = clearml_secret_preflight()
net_status = internet_check()
gpu_status = gpu_check()
if not gpu_status["gpu_available"]:
    raise RuntimeError("GPU is not available; stop before training.")
if not net_status["internet_ok"]:
    raise RuntimeError(f"Internet/ClearML host check failed: {net_status}")
INPUT_ROOT = resolve_input_root()
preflight = {"clearml_secret_status": secret_status, "internet": net_status, "gpu": gpu_status, "input_root": str(INPUT_ROOT)}
(OUTPUT_DIR / "notebook_08c_preflight.json").write_text(json.dumps(preflight, indent=2))
print("Preflight complete. Input root:", INPUT_ROOT)


In [ ]:
from clearml import Task
task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=f"{CLEARML_TASK_NAME_PREFIX}_{time.strftime('%Y%m%d_%H%M%S')}",
    task_type=Task.TaskTypes.training,
    auto_connect_frameworks=True,
    auto_connect_arg_parser=False,
)

def load_manifest(resolution):
    path = INPUT_ROOT / resolution / "notebook_04_split_windowing" / "selected_physics" / "window_ready_manifest.json"
    return json.loads(path.read_text())

def load_counts(resolution, lookback, horizon):
    path = INPUT_ROOT / resolution / "notebook_04_split_windowing" / "selected_physics" / "windowing_counts.json"
    counts = json.loads(path.read_text())["windowing_counts"]
    for item in counts:
        if int(item["lookback"]) == lookback and int(item["horizon"]) == horizon:
            return item
    raise KeyError((resolution, lookback, horizon))

def load_split_arrays(resolution):
    manifest = load_manifest(resolution)
    source = INPUT_ROOT / resolution / "notebook_03_physics_features" / "selected_features_plus_physics_informed_with_target.csv"
    scaler_file = INPUT_ROOT / resolution / "notebook_04_split_windowing" / "selected_physics" / "train_only_standard_scaler.joblib"
    df = pd.read_csv(source)
    features = manifest["feature_columns"]
    target_col = manifest["target_column"]
    if target_col != "ramp_rate_pu" or df.columns[-1] != "ramp_rate_pu":
        raise RuntimeError(f"Target alignment check failed for {resolution}")
    scaler = joblib.load(scaler_file)
    splits = {}
    for split_name, info in manifest["split_summary"].items():
        part = df.iloc[int(info["start_row_in_dataset"]):int(info["end_row_exclusive_in_dataset"])].copy()
        x = scaler.transform(part[features].to_numpy(dtype=np.float32)).astype("float32")
        y = part[target_col].to_numpy(dtype=np.float32)
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            raise RuntimeError(f"Non-finite values after scaling for {resolution}/{split_name}")
        splits[split_name] = {"x": x, "y": y}
    return splits, manifest

DATA_CACHE = {}
def get_data(resolution):
    if resolution not in DATA_CACHE:
        DATA_CACHE[resolution] = load_split_arrays(resolution)
    return DATA_CACHE[resolution]

def make_dataset(x, y, lookback, horizon, batch_size=256, shuffle=False):
    data = x[:-horizon]
    targets = y[lookback - 1 + horizon:]
    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=data,
        targets=targets,
        sequence_length=lookback,
        sequence_stride=1,
        sampling_rate=1,
        batch_size=batch_size,
        shuffle=shuffle,
    )
    return ds.prefetch(tf.data.AUTOTUNE), targets

def build_model(model_name, lookback, feature_count):
    inputs = keras.Input(shape=(lookback, feature_count))
    if model_name == "LSTM":
        x = layers.LSTM(64, dropout=0.10)(inputs)
    elif model_name == "GRU":
        x = layers.GRU(64, dropout=0.10)(inputs)
    elif model_name == "BiLSTM":
        x = layers.Bidirectional(layers.LSTM(64, dropout=0.10))(inputs)
    else:
        raise ValueError(model_name)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
    return model


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def r2_score_np(y_true, y_pred):
    denom = float(np.sum((y_true - np.mean(y_true)) ** 2))
    return float(1.0 - np.sum((y_true - y_pred) ** 2) / denom) if denom else float("nan")

def nrmse_np(y_true, y_pred):
    span = float(np.max(y_true) - np.min(y_true))
    return rmse(y_true, y_pred) / span if span else float("nan")

persistence_ref = pd.read_csv(INPUT_ROOT / "05_baseline_models" / "persistence_reference_by_variant_lh.csv")
hgb_summary = pd.read_csv(INPUT_ROOT / "07_classical_ml_models" / "ml_results_summary.csv")

def persistence_rmse_for(resolution, lookback, horizon, split):
    df = persistence_ref
    mask = (
        (df["resolution"] == resolution)
        & (df["dataset_variant"] == "selected_physics")
        & (df["lookback"] == lookback)
        & (df["horizon"] == horizon)
    )
    if "split" in df.columns:
        mask &= df["split"].astype(str).str.lower().eq(split)
    sub = df[mask]
    for col in ["persistence_rmse", "rmse", "RMSE", "persistence_rmse_reference"]:
        if len(sub) and col in sub.columns:
            return float(sub.iloc[0][col])
    return float("nan")

def hgb_reference_for(resolution, lookback, horizon):
    mask = (
        (hgb_summary["resolution"] == resolution)
        & (hgb_summary["dataset_variant"] == "selected_physics")
        & (hgb_summary["model"] == "HistGradientBoosting")
        & (hgb_summary["lookback"] == lookback)
        & (hgb_summary["horizon"] == horizon)
    )
    sub = hgb_summary[mask]
    if len(sub):
        return sub.iloc[0].to_dict()
    return {}

def evaluate_predictions(y_true, y_pred, persistence_rmse):
    y_pred = np.asarray(y_pred).reshape(-1)
    y_true = np.asarray(y_true).reshape(-1)
    value_rmse = rmse(y_true, y_pred)
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": value_rmse,
        "NRMSE": nrmse_np(y_true, y_pred),
        "R2": r2_score_np(y_true, y_pred),
        "Skill_RMSE_vs_Persistence": float(1.0 - value_rmse / persistence_rmse) if persistence_rmse and np.isfinite(persistence_rmse) else float("nan"),
    }


In [ ]:
all_split_rows, summary_rows, timing_rows, history_rows, skipped_rows = [], [], [], [], []

def save_partial():
    pd.DataFrame(all_split_rows).to_csv(OUTPUT_DIR / "dl_08c_results_train_validation_test.csv", index=False)
    pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / "dl_08c_results_summary.csv", index=False)
    pd.DataFrame(timing_rows).to_csv(OUTPUT_DIR / "dl_08c_timing_report.csv", index=False)
    pd.DataFrame(history_rows).to_csv(OUTPUT_DIR / "dl_08c_training_history.csv", index=False)
    pd.DataFrame(skipped_rows).to_csv(OUTPUT_DIR / "dl_08c_skipped_failed_runs.csv", index=False)

def run_config(config):
    run_key = f"{config['resolution']}__selected_physics__{config['model']}__L{config['lookback']}__H{config['horizon']}"
    state_path = RUN_STATE_DIR / f"{run_key}.json"
    if state_path.exists():
        state = json.loads(state_path.read_text())
        if state.get("status") == "success":
            print("Skipping completed config", run_key)
            return
    try:
        splits, manifest = get_data(config["resolution"])
        feature_count = int(manifest["feature_count"])
        train_ds, y_train = make_dataset(splits["train"]["x"], splits["train"]["y"], config["lookback"], config["horizon"], shuffle=True)
        val_ds, y_val = make_dataset(splits["validation"]["x"], splits["validation"]["y"], config["lookback"], config["horizon"], shuffle=False)
        test_ds, y_test = make_dataset(splits["test"]["x"], splits["test"]["y"], config["lookback"], config["horizon"], shuffle=False)
        model = build_model(config["model"], config["lookback"], feature_count)
        ckpt_path = CHECKPOINT_DIR / f"{run_key}.keras"
        callbacks = [
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(str(ckpt_path), monitor="val_loss", save_best_only=True),
        ]
        t0 = time.perf_counter()
        history = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=2)
        training_time = time.perf_counter() - t0
        best_epoch = int(np.argmin(history.history.get("val_loss", [np.nan])) + 1)
        total_epochs = int(len(history.history.get("loss", [])))
        epoch_times = training_time / total_epochs if total_epochs else float("nan")

        predictions = {}
        eval_times = {}
        truth = {"train": y_train, "validation": y_val, "test": y_test}
        datasets = {"train": train_ds, "validation": val_ds, "test": test_ds}
        for split_name, ds in datasets.items():
            start = time.perf_counter()
            pred = model.predict(ds, verbose=0).reshape(-1)
            eval_times[split_name] = time.perf_counter() - start
            predictions[split_name] = pred

        for epoch, values in enumerate(zip(history.history.get("loss", []), history.history.get("val_loss", [])), start=1):
            train_loss, val_loss = values
            history_rows.append({**config, "run_key": run_key, "epoch": epoch, "loss": train_loss, "val_loss": val_loss})
        hist_df = pd.DataFrame([r for r in history_rows if r["run_key"] == run_key])
        hist_df.to_csv(HISTORY_DIR / f"{run_key}_history.csv", index=False)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(history.history.get("loss", []), label="training_loss")
        ax.plot(history.history.get("val_loss", []), label="validation_loss")
        ax.set_title(run_key)
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.legend()
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / f"{run_key}_loss_curve.png", dpi=140)
        plt.close(fig)

        split_metrics = {}
        for split_name in ["train", "validation", "test"]:
            p_rmse = persistence_rmse_for(config["resolution"], config["lookback"], config["horizon"], split_name)
            m = evaluate_predictions(truth[split_name], predictions[split_name], p_rmse)
            split_metrics[split_name] = m
            all_split_rows.append({**config, "run_key": run_key, "split": split_name, **m, "sample_count": len(truth[split_name]), "persistence_rmse_reference": p_rmse, "training_time_seconds": training_time, "validation_time_seconds": eval_times["validation"], "test_time_seconds": eval_times["test"], "total_epochs_ran": total_epochs, "best_epoch": best_epoch, "early_stopping_used": True, "status": "success"})

        hgb_ref = hgb_reference_for(config["resolution"], config["lookback"], config["horizon"])
        summary_rows.append({**config, "run_key": run_key, "status": "success", "feature_count": feature_count, "train_RMSE": split_metrics["train"]["RMSE"], "validation_RMSE": split_metrics["validation"]["RMSE"], "test_RMSE": split_metrics["test"]["RMSE"], "validation_Skill_RMSE_vs_Persistence": split_metrics["validation"]["Skill_RMSE_vs_Persistence"], "test_Skill_RMSE_vs_Persistence": split_metrics["test"]["Skill_RMSE_vs_Persistence"], "hgb_validation_rmse_reference": hgb_ref.get("validation_rmse", np.nan), "hgb_test_rmse_reference": hgb_ref.get("test_rmse", np.nan), "training_time_seconds": training_time, "validation_time_seconds": eval_times["validation"], "test_time_seconds": eval_times["test"], "total_epochs_ran": total_epochs, "best_epoch": best_epoch, "early_stopping_used": True})
        timing_rows.append({**config, "run_key": run_key, "training_time_seconds": training_time, "validation_time_seconds": eval_times["validation"], "test_time_seconds": eval_times["test"], "epoch_time_mean_seconds": epoch_times, "total_epochs_ran": total_epochs, "best_epoch": best_epoch, "early_stopping_used": True})
        state_path.write_text(json.dumps({"run_key": run_key, "status": "success", "completed_at": time.strftime("%Y-%m-%dT%H:%M:%S")}, indent=2))
        save_partial()
    except Exception as exc:
        skipped_rows.append({**config, "run_key": run_key, "status": "failed", "skip_reason": str(exc), "traceback": traceback.format_exc()})
        state_path.write_text(json.dumps({"run_key": run_key, "status": "failed", "error": str(exc)}, indent=2))
        save_partial()
        print("Config failed but continuing:", run_key, exc)

for config in sorted(OFFICIAL_CONFIGS, key=lambda c: c["priority_order"]):
    run_config(config)
save_partial()


In [ ]:
summary_df = pd.read_csv(OUTPUT_DIR / "dl_08c_results_summary.csv") if (OUTPUT_DIR / "dl_08c_results_summary.csv").exists() else pd.DataFrame()
if len(summary_df):
    summary_df.to_csv(OUTPUT_DIR / "dl_08c_model_comparison.csv", index=False)
    summary_df.to_csv(OUTPUT_DIR / "dl_08c_vs_classical_benchmark.csv", index=False)
    summary_df.to_csv(OUTPUT_DIR / "dl_08c_performance_inference_tradeoff.csv", index=False)
    promoted = summary_df.sort_values(["horizon", "validation_RMSE"]).groupby(["resolution", "horizon"]).head(1)
    promoted.to_csv(OUTPUT_DIR / "dl_08c_promoted_candidates.csv", index=False)
else:
    pd.DataFrame().to_csv(OUTPUT_DIR / "dl_08c_model_comparison.csv", index=False)
    pd.DataFrame().to_csv(OUTPUT_DIR / "dl_08c_vs_classical_benchmark.csv", index=False)
    pd.DataFrame().to_csv(OUTPUT_DIR / "dl_08c_performance_inference_tradeoff.csv", index=False)
    pd.DataFrame().to_csv(OUTPUT_DIR / "dl_08c_promoted_candidates.csv", index=False)

final_report = {
    "status": "completed_with_possible_skips",
    "official_config_count": len(OFFICIAL_CONFIGS),
    "completed_count": int((summary_df.get("status", pd.Series(dtype=str)) == "success").sum()) if len(summary_df) else 0,
    "output_dir": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "notebook_08c_h8_h16_final_report.json").write_text(json.dumps(final_report, indent=2))
for artifact in [
    "dl_08c_results_train_validation_test.csv",
    "dl_08c_results_summary.csv",
    "dl_08c_timing_report.csv",
    "dl_08c_training_history.csv",
    "dl_08c_vs_classical_benchmark.csv",
    "dl_08c_model_comparison.csv",
    "dl_08c_performance_inference_tradeoff.csv",
    "dl_08c_skipped_failed_runs.csv",
    "dl_08c_promoted_candidates.csv",
    "notebook_08c_h8_h16_final_report.json",
]:
    p = OUTPUT_DIR / artifact
    if p.exists():
        task.upload_artifact(name=artifact, artifact_object=str(p))
zip_path = Path("/kaggle/working/notebook_08c_h8_h16_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))
task.upload_artifact(name="notebook_08c_h8_h16_outputs_zip", artifact_object=str(zip_path))
task.close()
print("Final output zip:", zip_path)
